# From Reads to a Taxonomy Table

> **Note:** like notebook 07 in the other course, the commands below are
> real but **not runnable in this browser** — QC tools, Bowtie2, and
> MetaPhlAn3 are compiled, conda-distributed software. Read this as a map
> of the real pipeline.

## Two sequencing strategies, one important difference

The other course's notebook 01 covers **16S rRNA sequencing**: sequence
one marker gene, cheap, genus-level resolution. GMWI2 needs something
different: **shotgun metagenomic sequencing** — sequence *everything* in
the sample, every gene, every organism, not just one marker. This is more
expensive and produces far more data per sample, but resolves down to
species (often strain) level, and lets you detect genes, not just
taxonomy — capabilities 16S data structurally cannot provide.

This is the mismatch flagged back in the other course's notebook 08: you
cannot point GMWI2 at 16S data. It's not a formatting problem — the two
methods measure fundamentally different things.

## The QC stage

Before any taxonomy assignment, raw shotgun reads go through the same
category of cleanup as any sequencing data:

```bash
# remove low-quality bases / adapter contamination
trimmomatic PE forward.fastq reverse.fastq \
  forward_clean.fastq forward_unpaired.fastq \
  reverse_clean.fastq reverse_unpaired.fastq \
  SLIDINGWINDOW:4:20 MINLEN:50

# remove human host DNA — inevitable contamination in a stool sample
bowtie2 -x GRCh38_index -1 forward_clean.fastq -2 reverse_clean.fastq \
  --un-conc host_removed_%.fastq -S /dev/null
```

The human-DNA removal step matters specifically for stool metagenomes:
a real sample is never 100% microbial — shed gut epithelial cells and
residual human DNA are always present, and every downstream percentage
(including a GMWI2 score) is implicitly "percent of the *microbial*
reads," not "percent of everything in the tube."

## MetaPhlAn3 — turning cleaned reads into a species table

MetaPhlAn3 matches reads against a curated database of marker genes —
short genetic sequences known to reliably identify a specific species —
rather than trying to assemble or classify every single read. This is
much faster than full alignment and is precisely why it can process
thousands of samples (recall: GMWI2's training set was 8,069 samples).

```bash
metaphlan forward_clean.fastq,reverse_clean.fastq \
  --bowtie2out sample.bowtie2.bz2 \
  --nproc 8 \
  --input_type fastq \
  -o sample_profile.txt
```

The output (`sample_profile.txt`) is a table: one row per taxon detected,
one column for relative abundance — structurally similar to the CSVs used
throughout the other course, except at species-level resolution instead
of genus, and produced from shotgun data instead of 16S.

### EXPLAIN #1
*MetaPhlAn3 identifies species using a curated set of marker genes rather
than every read in the sample. What's the tradeoff? What might it miss
compared to a method that tries to classify every single read?*

> your answer here

## Done — the table before the table

Every notebook from here on works from a species-level table like the one
MetaPhlAn3 produces. Notebook 03 picks up exactly there.

**Next:** `03_abundance_to_presence_absence.ipynb`.